# 02 · BM25 from scratch

After this notebook you can write BM25 from memory (inverted index, IDF, saturated term frequency, length normalisation), prove your version matches `rank_bm25`, explain every term of the formula with a picture, and say exactly which queries lexical search wins and loses against embeddings.

**Time:** ~30 min · **Runs:** offline on CPU in ~1 min · **Needs:** [chunking strategies](01_chunking_strategies_and_semantic_chunking.ipynb)

## Why this matters in interviews

- "What is BM25 and how does it work?" is asked in almost every RAG interview, and the follow-up is always **"why does it beat TF-IDF?"** - the answer is term-frequency saturation plus length normalisation, which you will plot.
- "Why is semantic search not enough?" has a concrete answer: **exact identifiers** (error codes, SKUs, model numbers). You will measure it on real queries.
- The analyzer (tokeniser) is where production lexical search silently breaks. Knowing one concrete example ("the `stop` filter deletes the `A` in `ERR-429-A`") is the kind of detail that signals hands-on experience.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ve08` | What is BM25 and how does it work? |
| `ve10` | What is an inverted index? |
| `ve11` | What is an analyzer, and what breaks when it is wrong? |
| `ve13` | Lexical versus dense retrieval - what is the honest comparison? |
| `rg02` | Why is embedding-based semantic search not enough for production RAG? |
| `ve09` | What is Elasticsearch and what is it actually good at? |
| `ve12` | Query context versus filter context - why does it matter? |

## Setup

Same Acme Cloud knowledge base and eval questions as every notebook in this section, split into one chunk per `##` section (heading path prepended).

In [ ]:
import math
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

np.random.seed(0)
pd.set_option("display.width", 150)
pd.set_option("display.max_colwidth", 60)

In [ ]:
# The knowledge base: 15 short markdown docs for a fictional company, Acme Cloud (a SaaS analytics product
# that also sells an 'Edge Gateway' device). This cell is data - it is identical in every 09_rag notebook.
KB = {
"refund_policy.md": """# Refund Policy

## Eligibility
Annual plans can be refunded in full within 30 days of purchase. Monthly plans can be cancelled at any time, but the current month is not refunded. Add-on credits and professional-services hours are non-refundable.

## How to request a refund
Open the billing portal, choose the invoice and click "Request refund". Only workspace owners can request refunds.

## Processing time
Approved refunds go back to the original payment method within 5-10 business days. Bank transfers can take up to 15 business days.
""",
"shipping.md": """# Hardware Shipping

## Delivery times
Edge Gateway devices ship from our warehouses in Rotterdam and Dallas. Standard delivery takes 3-5 business days. Express delivery takes 1-2 business days and costs $25.

## Tracking
A tracking link is emailed when the order leaves the warehouse. The first carrier scan can take 24 hours to appear.

## Lost or damaged shipments
If a shipment has not arrived 10 business days after dispatch, report it within 14 days so we can open a carrier investigation. We send a free replacement when the carrier confirms the loss.
""",
"warranty.md": """# Edge Gateway Warranty

## Coverage
Every Edge Gateway has a 2-year limited hardware warranty from the delivery date. It covers manufacturing defects in the device, the power supply and the antennas.

## Not covered
Water damage, drops, unauthorised repairs and cosmetic scratches are not covered by the warranty.

## RMA process
Request an RMA (return merchandise authorisation) in the support portal with the device serial number. Once the RMA is approved we ship a replacement within 7 days, and you send the faulty unit back with the prepaid label.
""",
"api_rate_limits.md": """# API Rate Limits

## Limits per plan
Free: 60 requests per minute. Pro: 600 requests per minute. Business: 1,200 requests per minute. Enterprise: 6,000 requests per minute. Limits apply per API key.

## Best practices
Retry with exponential backoff and jitter, cache responses you read often, and send writes through the bulk endpoint.
""",
"api_errors.md": """# API Error Reference

## ERR-429-A
The per-minute rate limit was exceeded. Wait for the number of seconds in the Retry-After header, then retry.

## ERR-429-B
The daily request quota is exhausted. The quota resets at 00:00 UTC; upgrade the plan for a higher quota.

## ERR-503-C
The API is in scheduled maintenance. Retry after the maintenance window shown on the status page.

## ERR-401-D
The API key was revoked or has expired. Create a new key in Settings > API keys.
""",
"gateway_models.md": """# Edge Gateway Models

## EG-100
The EG-100 is the entry model: 2 Ethernet ports, Wi-Fi 5 and an 8 W maximum power draw. It supports up to 50 sensors.

## EG-200
The EG-200 adds LTE backup: 4 Ethernet ports, Wi-Fi 6 and a 12 W maximum power draw. It supports up to 200 sensors.

## EG-300
The EG-300 is the rugged outdoor model: an IP67 enclosure, 4 Ethernet ports, Wi-Fi 6 and an 18 W maximum power draw. It supports up to 500 sensors.
""",
"oncall_policy.md": """# On-call Policy

## Rotation
Engineers join the on-call rotation after their third month. Each shift lasts one week and hands over on Monday at 09:00 local time. Every shift has a primary and a secondary.

## Response targets
A SEV1 page must be acknowledged within 5 minutes and a SEV2 page within 30 minutes. If a SEV1 page is not acknowledged within 15 minutes, it escalates to the secondary and then to the engineering manager.

## Compensation
On-call engineers receive a stipend of $300 per week, and a day off after any night with more than two pages.
""",
"pricing.md": """# Pricing Plans

## Plans
Free costs $0 and includes 3 users and 1 GB of storage. Pro costs $49 per user per month. Business costs $99 per user per month. Enterprise pricing is custom and starts at 50 users.

## What each plan includes
Single sign-on (SAML and OIDC) is included from the Business plan upward. Audit logs and the 99.95% uptime SLA are included only in Enterprise. All paid plans include email support.

## Billing
Paying annually saves 20% compared with monthly billing. Prices exclude VAT.
""",
"sso_setup.md": """# Single Sign-On Setup

## Supported identity providers
Acme Cloud supports Okta, Microsoft Entra ID (formerly Azure AD) and Google Workspace through SAML 2.0 or OIDC.

## Configuration steps
Go to Settings > Security > Single sign-on, upload the identity provider's metadata XML, and map the email and groups attributes. Test with one account before you enforce SSO for the whole workspace.

## Troubleshooting
SSO-ERR-12 means the SAML assertion was rejected because the clocks differ by more than 5 minutes; sync the identity provider's clock with NTP. SSO-ERR-31 means the email attribute is missing from the assertion.
""",
"data_retention.md": """# Data Retention

## Event data
Raw event data is kept for 30 days on Free, 13 months on Pro and Business, and 25 months on Enterprise. Aggregated reports are kept for as long as the workspace exists.

## Deleted workspaces
When a workspace is deleted, its data is purged from primary storage after 30 days and from backups after a further 35 days.

## Erasure requests
GDPR erasure requests for one person's data are completed within 30 days of identity verification.
""",
"pto_policy.md": """# Paid Time Off

## Allowance
Full-time employees receive 25 days of paid time off per calendar year, plus public holidays. Part-time employees receive a pro-rata allowance.

## Booking
Book absences longer than three days in Workday at least two weeks ahead. Your manager approves the request.

## Carry-over
Up to 5 unused days can be carried into the next year and must be used by 31 March.
""",
"webhooks.md": """# Webhooks

## Delivery and retries
Webhook events are sent as HTTPS POST requests. If your endpoint does not return a 2xx status within 10 seconds, we retry up to 8 times with exponential backoff over 24 hours.

## Verifying signatures
Every request carries an X-Acme-Signature header: an HMAC-SHA256 of the raw body computed with your webhook secret. Compare it in constant time and reject requests older than 5 minutes.
""",
"sla.md": """# Service Level Agreement

## Uptime commitment
Pro and Business are backed by a 99.9% monthly uptime commitment. Enterprise is backed by 99.95%.

## Service credits
If monthly uptime falls below the commitment, you receive a service credit of 10% of the monthly fee; below 99.0% the credit is 25%. Credits are applied to the next invoice.

## How to claim
Open a support ticket within 30 days of the end of the affected month and include the incident dates.
""",
"account_security.md": """# Account Security

## Passwords
Password reset links expire after 60 minutes. After 5 failed sign-in attempts the account is locked for 15 minutes.

## Two-factor authentication
Workspace owners can require two-factor authentication (2FA) for every member. Authenticator apps and hardware security keys are supported; SMS codes are not.
""",
"expenses.md": """# Travel and Expenses

## Meals and hotels
Meals are reimbursed up to $75 per day while travelling. Book hotels through the travel portal; the nightly cap is $200 in most cities.

## Submitting claims
Submit receipts in Workday within 30 days of the expense. Claims over $25 without a receipt are rejected.
""",
}

In [ ]:
# The eval set: each question lists its EVIDENCE - (doc, phrase) pairs that a retrieved chunk must contain.
# Phrases are unique in the KB, so "is the evidence in the retrieved text?" works for any chunking scheme.
QUESTIONS = [
    {"q": "What does ERR-429-A mean?", "kind": "exact id", "evidence": [("api_errors.md", "per-minute rate limit")]},
    {"q": "Our integration keeps failing with ERR-401-D", "kind": "exact id", "evidence": [("api_errors.md", "revoked")]},
    {"q": "How do I fix SSO-ERR-12?", "kind": "exact id", "evidence": [("sso_setup.md", "clocks differ")]},
    {"q": "How many sensors does the EG-300 support?", "kind": "exact id", "evidence": [("gateway_models.md", "500 sensors")]},
    {"q": "How do I get my money back after buying an annual plan by mistake?", "kind": "paraphrase",
     "evidence": [("refund_policy.md", "refunded in full within 30 days")]},
    {"q": "My parcel never showed up. What should I do?", "kind": "paraphrase", "evidence": [("shipping.md", "report it within 14 days")]},
    {"q": "How many vacation days do I get a year?", "kind": "paraphrase", "evidence": [("pto_policy.md", "25 days of paid time off")]},
    {"q": "Do we get compensated for outages?", "kind": "paraphrase", "evidence": [("sla.md", "service credit")]},
    {"q": "Which gateway can be installed outdoors?", "kind": "paraphrase", "evidence": [("gateway_models.md", "rugged outdoor model")]},
    {"q": "Who gets paged when nobody acknowledges a SEV1 alert?", "kind": "keyword", "evidence": [("oncall_policy.md", "engineering manager")]},
    {"q": "How long is event data kept on the Pro plan?", "kind": "keyword", "evidence": [("data_retention.md", "13 months on Pro")]},
    {"q": "What do we get back if the platform is down more than promised?", "kind": "vague", "evidence": [("sla.md", "service credit")]},
    {"q": "What is the API rate limit on the cheapest plan that includes SSO?", "kind": "multi-hop",
     "evidence": [("pricing.md", "from the Business plan upward"), ("api_rate_limits.md", "Business: 1,200 requests")]},
    {"q": "How long do approved refunds take, and how long is the gateway warranty?", "kind": "multi-part",
     "evidence": [("refund_policy.md", "5-10 business days"), ("warranty.md", "2-year")]},
    {"q": "What is the capital of Australia?", "kind": "out-of-scope", "evidence": []},
]
IN_SCOPE = [q for q in QUESTIONS if q["evidence"]]


def has_evidence(text, doc, q):
    """True if this text (from `doc`) contains any of the question's evidence phrases."""
    return any(doc == d and p.lower() in text.lower() for d, p in q["evidence"])


print(f"{len(KB)} docs, {sum(len(t) for t in KB.values()):,} characters; "
      f"{len(QUESTIONS)} questions ({len(IN_SCOPE)} answerable, {len(QUESTIONS) - len(IN_SCOPE)} out of scope)")

In [ ]:
import re


def split_sections(kb):
    """One chunk per '## ' section. The heading path is prepended so every chunk says where it came from."""
    chunks = []
    for doc, text in kb.items():
        title = text.splitlines()[0].lstrip("# ").strip()
        for i, part in enumerate(re.split(r"^## ", text, flags=re.M)[1:]):
            heading, _, body = part.partition("\n")
            path = f"{title} > {heading.strip()}"
            chunks.append({"id": f"{doc}#{i}", "doc": doc, "section": path, "text": f"{path}\n{body.strip()}"})
    return chunks


def is_relevant(chunk, q):
    return has_evidence(chunk["text"], chunk["doc"], q)


CHUNKS = split_sections(KB)
print(f"{len(CHUNKS)} section chunks, avg {sum(len(c['text']) for c in CHUNKS) / len(CHUNKS):.0f} characters")
print("example:", CHUNKS[1]["id"], "->", CHUNKS[1]["text"][:90].replace("\n", " | "), "...")

## 1. The analyzer: text in, terms out

Lexical search never sees text, only **terms**. Our default analyzer: lowercase, keep hyphenated identifiers like `err-429-a` as one term, drop stopwords. (Section 5 shows what goes wrong with other choices.)

In [ ]:
STOPWORDS = set("""a an the is are was were be been of to in on for with and or but if then so as at by from this that
it its i you we my our your do does did can how what who when which why there their they me""".split())


def tokenize(text):
    return [t for t in re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower()) if t not in STOPWORDS]


print(tokenize("What does ERR-429-A mean? Retry after the Retry-After header."))
corpus_tokens = [tokenize(c["text"]) for c in CHUNKS]
print(f"{len(corpus_tokens)} chunks, avg length {np.mean([len(t) for t in corpus_tokens]):.1f} terms, "
      f"vocabulary {len(set(t for toks in corpus_tokens for t in toks))} terms")

## 2. TF-IDF recap - and the flaw BM25 fixes

TF-IDF scores a document by summing, over query terms, **term frequency × inverse document frequency**:

$$\text{tfidf}(q, d) = \sum_{t \in q} tf(t, d) \cdot \log\frac{N}{df(t)}$$

- **tf** - how often the term is in this chunk (more = more relevant... up to a point)
- **idf** - how rare the term is across the corpus (`err-429-a` in 1 of 42 chunks is worth far more than `days` in 10)

The flaw is the word *linearly*: 40 mentions count 40 times as much as one. Add a spam chunk that just repeats "refund" and watch.

In [ ]:
def tfidf_scores(query_tokens, docs_tokens):
    N = len(docs_tokens)
    df = Counter(t for toks in docs_tokens for t in set(toks))
    scores = np.zeros(N)
    for i, toks in enumerate(docs_tokens):
        tf = Counter(toks)
        scores[i] = sum(tf[t] * math.log(N / df[t]) for t in query_tokens if df[t])
    return scores


spam = "Refund refund refund! " * 14 + "Best refund offers, refund now."
docs_with_spam = corpus_tokens + [tokenize(spam)]
names = [c["id"] for c in CHUNKS] + ["SPAM"]
query = tokenize("refund processing time")

tfidf = tfidf_scores(query, docs_with_spam)
top = np.argsort(-tfidf)[:3]
print("query terms:", query)
print("TF-IDF top 3:", [(names[i], round(float(tfidf[i]), 2)) for i in top])
assert names[top[0]] == "SPAM", "linear TF lets keyword stuffing win"

The spam chunk wins because 45 mentions of "refund" outweigh a chunk that actually talks about refund *processing time*. Real search engines stopped using raw TF-IDF for exactly this reason.

## 3. BM25, term by term

$$\text{BM25}(q, d) = \sum_{t \in q} \underbrace{\text{IDF}(t)}_{\text{rare terms count more}} \cdot \underbrace{\frac{tf \cdot (k_1 + 1)}{tf + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}}_{\text{saturating TF, normalised by length}}$$

Three ideas, each with a picture:

1. **IDF** - $\log\frac{N - n + 0.5}{n + 0.5}$ (classic Okapi; $n$ = chunks containing the term). Lucene/Elasticsearch use $\log\left(1 + \frac{N - n + 0.5}{n + 0.5}\right)$ so it never goes negative.
2. **TF saturation, controlled by $k_1$** - the 1st mention matters a lot, the 40th almost nothing. The TF part can never exceed $k_1 + 1$.
3. **Length normalisation, controlled by $b$** - a term found in a long chunk counts less than in a short one ($b = 0$: ignore length; $b = 1$: fully normalise).

In [ ]:
N = len(CHUNKS)
n = np.arange(1, N + 1)
idf_okapi = np.log((N - n + 0.5) / (n + 0.5))
idf_lucene = np.log(1 + (N - n + 0.5) / (n + 0.5))

plt.figure(figsize=(8, 3.6))
plt.plot(n, idf_okapi, label="Okapi: log((N-n+0.5)/(n+0.5))", color="#3b5bdb")
plt.plot(n, idf_lucene, label="Lucene: log(1 + (N-n+0.5)/(n+0.5))", color="#e8590c")
plt.axhline(0, color="black", lw=0.8)
plt.axvline(N / 2, color="#adb5bd", ls=":", label="term in half the chunks")
plt.xlabel(f"n = number of chunks containing the term (N = {N})")
plt.ylabel("IDF")
plt.title("IDF: rare terms dominate; common terms go to ~0 (or below!)")
plt.legend()
plt.tight_layout()
plt.show()

Okapi IDF goes **negative** for a term in more than half the chunks - matching it would *lower* the score. `rank_bm25` patches this by replacing negative IDFs with `epsilon × average IDF`; Lucene adds 1 inside the log. Small corpora hit this constantly.

In [ ]:
tf = np.arange(0, 21)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for k1, color in zip([0.5, 1.2, 2.0, 5.0], ["#74c0fc", "#3b5bdb", "#e8590c", "#2b8a3e"]):
    ax1.plot(tf, tf * (k1 + 1) / (tf + k1), label=f"k1 = {k1}", color=color)
ax1.plot(tf, tf / 4, "k--", lw=1, label="plain TF (linear, scaled)")
ax1.set_xlabel("term frequency in the chunk")
ax1.set_ylabel("TF component (b = 0)")
ax1.set_title("Saturation: the ceiling is k1 + 1")
ax1.legend()

ratio = np.linspace(0.2, 3, 60)
for b, color in zip([0.0, 0.5, 0.75, 1.0], ["#adb5bd", "#74c0fc", "#3b5bdb", "#e8590c"]):
    ax2.plot(ratio, 3 * 2.2 / (3 + 1.2 * (1 - b + b * ratio)), label=f"b = {b}", color=color)
ax2.axvline(1, color="black", lw=0.8, ls=":")
ax2.set_xlabel("chunk length / average length")
ax2.set_ylabel("TF component (tf = 3, k1 = 1.2)")
ax2.set_title("Length normalisation: long chunks are discounted")
ax2.legend()
plt.tight_layout()
plt.show()

Read the two panels together: with the default $k_1 \approx 1.2$-$1.5$, going from 1 to 2 mentions is a big jump, from 10 to 20 almost nothing - keyword stuffing stops paying. And a chunk twice the average length needs noticeably more mentions to score the same, which stops long chunks winning just by containing more words.

## 4. Implement BM25 with an inverted index

An **inverted index** maps each term to its *posting list*: the chunks containing it and how often. Scoring only touches the posting lists of the query terms - that is why lexical search scales to billions of documents.

In [ ]:
class BM25:
    def __init__(self, docs_tokens, k1=1.5, b=0.75, epsilon=0.25, idf_variant="okapi"):
        self.k1, self.b = k1, b
        self.N = len(docs_tokens)
        self.doc_len = np.array([len(toks) for toks in docs_tokens], dtype=float)
        self.avgdl = self.doc_len.mean()
        self.postings = {}                                   # term -> {doc_id: tf}
        for doc_id, toks in enumerate(docs_tokens):
            for term, tf in Counter(toks).items():
                self.postings.setdefault(term, {})[doc_id] = tf
        self.idf = {}
        for term, plist in self.postings.items():
            n = len(plist)
            if idf_variant == "lucene":
                self.idf[term] = math.log(1 + (self.N - n + 0.5) / (n + 0.5))
            else:
                self.idf[term] = math.log((self.N - n + 0.5) / (n + 0.5))
        if idf_variant == "okapi":                           # rank_bm25's floor for negative IDF
            floor = epsilon * (sum(self.idf.values()) / len(self.idf))
            self.idf = {t: (v if v >= 0 else floor) for t, v in self.idf.items()}

    def term_score(self, term, doc_id):
        tf = self.postings.get(term, {}).get(doc_id, 0)
        norm = self.k1 * (1 - self.b + self.b * self.doc_len[doc_id] / self.avgdl)
        return self.idf.get(term, 0.0) * tf * (self.k1 + 1) / (tf + norm)

    def get_scores(self, query_tokens):
        scores = np.zeros(self.N)
        for term in query_tokens:                            # only walk the posting lists we need
            for doc_id in self.postings.get(term, {}):
                scores[doc_id] += self.term_score(term, doc_id)
        return scores

    def explain(self, query_tokens, doc_id):
        return {t: round(float(self.term_score(t, doc_id)), 3) for t in query_tokens}


bm25 = BM25(corpus_tokens)
for term in ["err-429-a", "refunds", "days"]:
    print(f"posting list for {term!r:12}: {{chunk: tf}} = {bm25.postings.get(term)}  idf={bm25.idf.get(term, 0):.2f}")

Now the claim that matters: **our scores equal the reference implementation's** on every eval question.

In [ ]:
from rank_bm25 import BM25Okapi

reference = BM25Okapi(corpus_tokens)                         # same defaults: k1=1.5, b=0.75, epsilon=0.25
for q in QUESTIONS:
    ours, theirs = bm25.get_scores(tokenize(q["q"])), reference.get_scores(tokenize(q["q"]))
    assert np.allclose(ours, theirs), q["q"]
print(f"our BM25 == rank_bm25.BM25Okapi on all {len(QUESTIONS)} questions")

spam_bm25 = BM25(docs_with_spam)
s = spam_bm25.get_scores(query)
top = np.argsort(-s)[:3]
print("BM25 top 3 with the spam chunk:", [(names[i], round(float(s[i]), 2)) for i in top])
print("spam 'refund' contribution:", spam_bm25.explain(query, len(names) - 1))
print("real chunk contributions  :", spam_bm25.explain(query, names.index("refund_policy.md#2")))
assert names[top[0]] == "refund_policy.md#2", "saturation + length normalisation defeat keyword stuffing"

Same spam, opposite verdict: the spam chunk's 45 mentions of "refund" are capped near $(k_1+1) \cdot \text{IDF}$ and discounted for its length, while the real "Processing time" chunk collects points from the rarer terms *processing* and *time*. `explain` is the per-term breakdown Elasticsearch gives you with `"explain": true` - lexical scores are fully explainable, a cosine of 0.83 is not.

## 5. Tokenisation choices change the results

Same BM25, five analyzers. For each query we report the rank of the first chunk that holds the evidence (lower is better; `-` = never retrieved).

In [ ]:
def simple_stem(w):
    for suffix, repl in (("ies", "y"), ("ing", ""), ("ed", ""), ("es", ""), ("s", "")):
        if len(w) > len(suffix) + 3 and w.endswith(suffix):
            return w[: -len(suffix)] + repl
    return w


ANALYZERS = {
    "split on spaces, keep case": lambda t: t.split(),
    "lowercase words": lambda t: re.findall(r"[a-z0-9]+", t.lower()),
    "+ stopwords": lambda t: [w for w in re.findall(r"[a-z0-9]+", t.lower()) if w not in STOPWORDS],
    "+ stopwords + stemming": lambda t: [simple_stem(w) for w in re.findall(r"[a-z0-9]+", t.lower()) if w not in STOPWORDS],
    "keep IDs whole + stopwords (ours)": tokenize,
}
probe_queries = [
    {"q": "What does ERR-429-A mean?", "evidence": [("api_errors.md", "per-minute rate limit")]},
    {"q": "err-429-a", "evidence": [("api_errors.md", "per-minute rate limit")]},
    {"q": "refunds for annual plan", "evidence": [("refund_policy.md", "refunded in full within 30 days")]},
    {"q": "Which gateway can be installed outdoors?", "evidence": [("gateway_models.md", "rugged outdoor model")]},
]


def first_relevant_rank(scores, q):
    for rank, i in enumerate(np.argsort(-scores, kind="stable"), 1):
        if scores[i] <= 0:
            return None
        if is_relevant(CHUNKS[i], q):
            return rank
    return None


rows = []
for q in probe_queries:
    row = {"query": q["q"]}
    for name, analyzer in ANALYZERS.items():
        index = BM25([analyzer(c["text"]) for c in CHUNKS])
        r = first_relevant_rank(index.get_scores(analyzer(q["q"])), q)
        row[name] = r if r else "-"
    rows.append(row)
table = pd.DataFrame(rows).set_index("query")
table.columns = ["spaces+case", "lowercase", "+stopwords", "+stop+stem", "IDs whole (ours)"]
print(table.to_string())

stop_an = ANALYZERS["+ stopwords"]
stop_index = BM25([stop_an(c["text"]) for c in CHUNKS])
ids = [c["id"] for c in CHUNKS]
q_terms = stop_an("What does ERR-429-A mean?")
print("\n'+ stopwords' turns the query into", q_terms, "- the 'a' is gone, so the code cannot tell A from B:")
for cid in ["api_errors.md#0", "api_errors.md#1"]:
    print(f"  {cid} ({CHUNKS[ids.index(cid)]['section']}): {stop_index.explain(q_terms, ids.index(cid))}")
assert "a" not in q_terms and tokenize("ERR-429-A") == ["err-429-a"]

What the table shows (read each column top to bottom):

- **Case-sensitive splitting** fails on anything with punctuation or case differences ("mean?" is not "mean", "ERR-429-A" never equals "err-429-a").
- **Splitting on hyphens + stopwords** turns `ERR-429-A` into `["err", "429"]` - the stopword filter deleted the `a`, so the ERR-429-A and ERR-429-B chunks are scored on the same two terms and the order between them is decided by chunk length, not by the code. This is exactly what Elasticsearch's `standard` tokenizer + `stop` filter does to codes.
- **Stemming** lets "refunds ... plan" match "refunded ... plans" - more recall, occasionally less precision.
- **Keeping IDs whole** makes the code a single, very rare term with a huge IDF.

The production fix is to index both forms (e.g. Elasticsearch `word_delimiter_graph` with `preserve_original: true`, or a second `keyword` sub-field for codes) and, above all, run **the same analyzer at index and query time**.

## 6. Where BM25 beats dense retrieval - and where it loses

Run both on every eval question: BM25 (our class) and dense cosine with MiniLM embeddings.

In [ ]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import transformers
transformers.utils.logging.disable_progress_bar()          # keep the cell output clean
transformers.logging.set_verbosity_error()
from sentence_transformers import SentenceTransformer


def load_st(name, cls=SentenceTransformer):
    """Load a small Hugging Face model: cached copy first (no network), download only if missing."""
    try:
        return cls(name, local_files_only=True)
    except Exception:
        try:
            return cls(name)                                 # first run only: ~90 MB download
        except Exception as e:
            raise RuntimeError(f"Could not load {name}: connect to the internet once so it can download.") from e


embedder = load_st("sentence-transformers/all-MiniLM-L6-v2")


def encode(texts):
    """Unit-length MiniLM embeddings (384-d), so a dot product IS the cosine similarity."""
    return embedder.encode(list(texts), normalize_embeddings=True, convert_to_numpy=True)


print("embedder: all-MiniLM-L6-v2 |", embedder.get_embedding_dimension(), "dims | max",
      embedder.max_seq_length, "tokens per text (the rest is silently cut off)")

In [ ]:
chunk_vecs = encode([c["text"] for c in CHUNKS])
rows = []
for q in IN_SCOPE:
    r_bm25 = first_relevant_rank(bm25.get_scores(tokenize(q["q"])), q)
    dense = chunk_vecs @ encode([q["q"]])[0]
    r_dense = next(r for r, i in enumerate(np.argsort(-dense), 1) if is_relevant(CHUNKS[i], q))
    winner = "tie" if r_bm25 == r_dense else ("BM25" if r_bm25 and r_bm25 < r_dense else "dense")
    rows.append({"kind": q["kind"], "question": q["q"], "BM25 rank": r_bm25 or "-", "dense rank": r_dense, "winner": winner})
cmp = pd.DataFrame(rows)
print(cmp.to_string(index=False))

for name in ["BM25 rank", "dense rank"]:
    ranks = [r if r != "-" else 10**9 for r in cmp[name]]
    print(f"{name:10s}: hit@1 = {np.mean([r <= 1 for r in ranks]):.2f}  hit@3 = {np.mean([r <= 3 for r in ranks]):.2f}")

by_q = cmp.set_index("question")
assert by_q.loc["What does ERR-429-A mean?", "BM25 rank"] == 1 and by_q.loc["What does ERR-429-A mean?", "dense rank"] > 1
assert by_q.loc["My parcel never showed up. What should I do?", "BM25 rank"] == "-"
assert by_q.loc["My parcel never showed up. What should I do?", "dense rank"] == 1

In [ ]:
kinds = cmp["kind"].unique()
wins = pd.crosstab(cmp["kind"], cmp["winner"]).reindex(index=kinds, columns=["BM25", "tie", "dense"], fill_value=0)
wins.plot(kind="bar", stacked=True, color=["#e8590c", "#adb5bd", "#3b5bdb"], figsize=(8, 3.6), rot=0)
plt.ylabel("questions")
plt.title("Who ranks the evidence higher, by question type")
plt.tight_layout()
plt.show()

The pattern is the one every production team rediscovers:

| Query type | Winner | Why |
|---|---|---|
| Exact identifiers (`ERR-429-A`, `EG-300`, `SSO-ERR-12`) | **BM25**, or a tie | one rare term with a huge IDF; to an embedding, `429-A` and `429-B` look alike (dense put ERR-429-B first) |
| Paraphrase / synonyms ("parcel never showed up" vs "shipment has not arrived") | **dense** | BM25 needs the same words; here it matched nothing useful |
| Plain keyword questions | usually a tie | both find it |
| Vague ("what do we get back if the platform is down...") | neither | a query-rewriting problem ([notebook 07](07_advanced_rag_query_rewriting_hyde_multiquery.ipynb)) |

Neither lane dominates, and they fail on *different* questions - which is the whole argument for running both and fusing them ([notebook 04](04_hybrid_search_and_rrf.ipynb)).

## 7. BM25 in Elasticsearch / OpenSearch

Everything above is what Elasticsearch/OpenSearch (both built on Lucene) do for you, at scale. BM25 is the **default similarity** (k1 = 1.2, b = 0.75). What you configure is the analyzer, the similarity parameters and the query shape:

```json
PUT /acme-kb
{
  "settings": {
    "index": { "similarity": { "kb_bm25": { "type": "BM25", "k1": 1.2, "b": 0.75 } } },
    "analysis": { "analyzer": { "kb_text": { "type": "custom", "tokenizer": "standard",
                                             "filter": ["lowercase", "stop", "porter_stem"] } } }
  },
  "mappings": { "properties": {
    "text":   { "type": "text", "analyzer": "kb_text", "similarity": "kb_bm25",
                "fields": { "exact": { "type": "keyword" } } },
    "doc":    { "type": "keyword" },
    "tenant": { "type": "keyword" }
  } }
}

GET /acme-kb/_analyze
{ "analyzer": "kb_text", "text": "ERR-429-A means the per-minute limit" }
// -> err, 429, per, minut, limit   (the "a" is removed by the stop filter - section 5's bug)

GET /acme-kb/_search
{
  "query": { "bool": {
    "must":   [ { "match": { "text": "What does ERR-429-A mean?" } } ],
    "filter": [ { "term": { "tenant": "acme" } } ]
  } },
  "explain": true
}
```

- **`must` = query context** (scored with BM25); **`filter` = filter context** (yes/no, not scored, cached as a bitset - faster). Tenant and permission checks belong in `filter`: a permission expressed as a score is not a permission.
- `_analyze` shows the terms an analyzer produces - the first thing to run when "nothing matches".
- Lucene (≥ 8) uses the non-negative IDF from section 3 and drops the constant $(k_1+1)$ factor; that rescales every score equally, so rankings are unchanged.
- Both engines also store dense vectors (`dense_vector` / `knn_vector`) and can fuse the two lanes (Elasticsearch has an `rrf` retriever; OpenSearch a hybrid query with score normalisation), which makes hybrid search one system instead of two.

## Try it yourself

**Exercise 1.** Tune BM25: sweep `k1` in {0.5, 1.2, 2.0} and `b` in {0, 0.75, 1.0}; report MRR of the first relevant chunk on the eval set. Are the defaults close to the best?

In [ ]:
# Solution — try it yourself first, then run this
def mrr(index):
    rr = []
    for q in IN_SCOPE:
        r = first_relevant_rank(index.get_scores(tokenize(q["q"])), q)
        rr.append(1 / r if r else 0.0)
    return np.mean(rr)


grid = pd.DataFrame([[mrr(BM25(corpus_tokens, k1=k1, b=b)) for b in (0.0, 0.75, 1.0)] for k1 in (0.5, 1.2, 2.0)],
                    index=["k1=0.5", "k1=1.2", "k1=2.0"], columns=["b=0", "b=0.75", "b=1"])
print(grid.round(3))

**Exercise 2.** Negative IDF in practice: build BM25 over only the three refund-policy chunks, list the terms that appear in more than half of them, and compare their raw Okapi, floored Okapi and Lucene IDFs.

In [ ]:
# Solution — try it yourself first, then run this
small = [tokenize(c["text"]) for c in CHUNKS if c["doc"] == "refund_policy.md"]
okapi, lucene = BM25(small), BM25(small, idf_variant="lucene")
for t, plist in okapi.postings.items():
    if len(plist) > len(small) / 2:
        raw = math.log((len(small) - len(plist) + 0.5) / (len(plist) + 0.5))
        print(f"{t!r:10}: in {len(plist)}/3 chunks | okapi raw {raw:+.3f} -> floored {okapi.idf[t]:+.3f} | lucene {lucene.idf[t]:+.3f}")
most_common = max(bm25.postings, key=lambda t: len(bm25.postings[t]))
print(f"full corpus: the most common term, {most_common!r}, is in {len(bm25.postings[most_common])}/{bm25.N} chunks "
      "- below half, so the floor never triggers there")

**Exercise 3.** Query-time synonym expansion: add a tiny synonym map (`parcel -> shipment`, `outages -> uptime`, `vacation -> "paid time off"`) and re-run BM25 on the questions BM25 missed. Which ones does it rescue?

In [ ]:
# Solution — try it yourself first, then run this
SYNONYMS = {"parcel": "shipment arrived", "outages": "uptime service credit", "vacation": "paid time off",
            "money back": "refund refunded"}


def expand(query):
    extra = [v for k, v in SYNONYMS.items() if k in query.lower()]
    return query + " " + " ".join(extra)


for q in IN_SCOPE:
    before = first_relevant_rank(bm25.get_scores(tokenize(q["q"])), q)
    after = first_relevant_rank(bm25.get_scores(tokenize(expand(q["q"]))), q)
    if before != after:
        print(f"{q['q'][:55]:57s} rank {before or '-'} -> {after or '-'}")

## Say it in an interview

- **30 seconds:** "BM25 scores a document by summing, over query terms, IDF times a saturating term frequency normalised by document length. IDF makes rare terms like error codes dominate; saturation, set by k1 around 1.2, means the 50th mention adds almost nothing, which is what beats TF-IDF and keyword stuffing; b around 0.75 discounts long documents. It runs on an inverted index, so it only touches the posting lists of the query terms."
- **Honest comparison:** BM25 wins exact identifiers, new jargon and anything that must match literally, is explainable and needs no model; dense wins paraphrases and synonyms. They fail on different queries, so production runs both and fuses by rank.
- **Numbers:** k1 in 1.2-2.0, b = 0.75; ES default k1 = 1.2, b = 0.75; Okapi IDF goes negative for terms in more than half the documents.
- **Traps:** a different analyzer at index and query time (every query runs, nothing matches, no error); the `stop` filter eating part of a code; a tenant filter in query context instead of filter context; claiming BM25 "understands" anything.
- **Follow-ups:** "what is SPLADE?" (learned sparse vectors on an inverted index - BM25 that learned synonyms); "how does ES scale it?" (shards of immutable Lucene segments, near-real-time refresh ~1 s).

## Next

- [Vector search with FAISS and HNSW](03_vector_search_faiss_hnsw.ipynb) - the dense lane at scale.
- [Hybrid search and RRF](04_hybrid_search_and_rrf.ipynb) - fuse the two lanes so each covers the other's failures.
- [Chunking strategies](01_chunking_strategies_and_semantic_chunking.ipynb) · [RAG pipeline from scratch](06_rag_pipeline_from_scratch.ipynb)
- Theory: [genai_flow course](../../genai_flow/index.html) · [AI system design concepts](../../ai_system_design_concepts/index.html) · [interview bank](../../ai_interview_prep/index.html)